In [1]:
"""
Peningkatan citra MRI: Gaussian Two-Scale Decomposition + CLAHE + LED + MPA
Menjalankan Skenario B, C (metode usulan), dan D. Skenario A (baseline Dinh, 2023)
TIDAK dijalankan ulang; angkanya diambil dari Tabel 9 Dinh (2023).

Dependensi: numpy, opencv-python, scipy, scikit-image, pandas

REVISI (dibanding versi sebelumnya):
  1. apply_led(): normalisasi SELALU dilakukan (dibagi nilai maksimum) di semua skenario,
     sehingga skala I_2 konsisten dan bobot w2 antar-skenario dapat dibandingkan.
  2. Skenario D didokumentasikan eksplisit: I_2 = I_detail mentah (bertanda +/-), tanpa LED,
     tetap dikalikan w2 lalu dijumlahkan dengan w1 * CLAHE(I_base).
  3. PSNR memakai L = nilai maksimum piksel citra masukan (mengikuti Dinh, 2023),
     bukan L = 1 tetap. Baris NE tidak lagi memuat SSIM/PSNR/waktu (dibandingkan dengan
     dirinya sendiri, sehingga tidak bermakna).
  4. Seed acak diatur ulang per (citra, skenario) agar B, C, D memakai aliran acak yang
     sama dan hasil tidak bergantung pada urutan eksekusi.

CATATAN PARAMETER (isi di Bab III):
  - clipLimit = 2.0 dan tileGridSize = (8, 8) adalah PILIHAN PENULIS (nilai bawaan OpenCV),
    tidak disebut pada Dinh (2023), Liu et al. (2021), maupun Kieu et al. (2024).
  - Kernel Laplacian 4-tetangga juga pilihan penulis.
  - Dataset M1: irisan ke-55 s.d. ke-84 (bukan 50-79 milik Kieu et al.).
"""
import os
import time
import glob
import math
import cv2
import numpy as np
import pandas as pd
from scipy.signal import convolve2d
from skimage.metrics import structural_similarity as compute_ssim


# ==============================================================================
# 1. KONFIGURASI & PARAMETER EKSPERIMEN
# ==============================================================================
CONFIG = {
    # Dataset M1: 90 citra MRI (irisan 55-84 untuk sumbu T, S, C), 256x256
    "dataset_dir": "./Dataset_M1_MRI",
    "output_dir": "./Hasil_Eksperimen",
    "image_size": (256, 256),

    # CLAHE: pilihan penulis (bukan dari paper acuan)
    "clahe_clip_limit": 2.0,
    "clahe_tile_grid_size": (8, 8),

    # Kernel Laplacian 3x3 (4-tetangga): pilihan penulis
    "laplacian_kernel": np.array([
        [0,  1, 0],
        [1, -4, 1],
        [0,  1, 0]
    ], dtype=np.float64),

    # Kernel Gaussian 5x5 (sigma = 1) sesuai Kieu et al. (2024)
    "gaussian_kernel": (1.0 / 330.0) * np.array([
        [1,  4,  7,  4, 1],
        [4, 20, 33, 20, 4],
        [7, 33, 54, 33, 7],
        [4, 20, 33, 20, 4],
        [1,  4,  7,  4, 1]
    ], dtype=np.float64),

    # Hiperparameter MPA sesuai Dinh (2023)
    "mpa_n_prey": 50,
    "mpa_max_iter": 50,
    "mpa_K": 0.5,
    "mpa_FADs": 0.2,

    # Hipotesis awal rentang bobot (w1, w2); validasi lewat uji sensitivitas
    "bounds_weights": {
        "lb": np.array([0.5, 0.1]),
        "ub": np.array([1.5, 0.5])
    },

    # False = Skema Replikasi (E1 - E2), (G1 - G2) seperti persamaan asli Dinh (2023)
    # True  = Skema Modifikasi (E2 - E1), (G2 - G1)
    "use_modified_fitness": False,
    "random_seed": 42
}


# ==============================================================================
# 2. METRIK EVALUASI & FUNGSI OBJEKTIF
# ==============================================================================
def pixel_max(img: np.ndarray) -> float:
    """Nilai maksimum piksel citra (L pada PSNR, mengikuti Dinh, 2023)."""
    m = float(np.max(img))
    return m if m > 1e-12 else 1.0


def compute_entropy(img_01: np.ndarray) -> float:
    """Entropi citra (log2, 256 level) sesuai Tabel 2 Dinh (2023)."""
    img_uint8 = np.clip(np.round(img_01 * 255.0), 0, 255).astype(np.uint8)
    hist = cv2.calcHist([img_uint8], [0], None, [256], [0, 256]).ravel()
    prob = hist / hist.sum()
    prob = prob[prob > 0]
    return float(-np.sum(prob * np.log2(prob)))


def compute_average_gradient(img_01: np.ndarray) -> float:
    """Average Gradient (AG) sesuai Tabel 2 Dinh (2023)."""
    gx = img_01[1:, :-1] - img_01[:-1, :-1]
    gy = img_01[:-1, 1:] - img_01[:-1, :-1]
    return float(np.mean(np.sqrt(gx ** 2 + gy ** 2)))


def compute_psnr(img_enhanced: np.ndarray, img_ref: np.ndarray, max_val: float) -> float:
    """PSNR = 10 log10(L^2 / MSE). L ditentukan pemanggil (lihat pixel_max)."""
    mse = np.mean((img_ref - img_enhanced) ** 2)
    if mse <= 1e-12:
        return 100.0
    return float(10.0 * np.log10((max_val ** 2) / mse))


def estimate_noise_sigma(img_01: np.ndarray) -> float:
    """Estimasi sigma noise (metode Immerkaer) untuk memantau amplifikasi noise."""
    H, W = img_01.shape
    kernel_noise = np.array([
        [ 1, -2,  1],
        [-2,  4, -2],
        [ 1, -2,  1]
    ], dtype=np.float64)
    conv = convolve2d(img_01, kernel_noise, mode='valid')
    sigma = np.sum(np.abs(conv)) * np.sqrt(0.5 * np.pi) / (6.0 * (W - 2) * (H - 2))
    return float(sigma)


def evaluate_all_metrics(img_enhanced: np.ndarray, img_ref: np.ndarray) -> dict:
    """
    MLI, CI, Entropy, AG  -> dibandingkan dengan Tabel 9 Dinh (2023)
    SSIM, PSNR, Noise_Est -> hanya untuk perbandingan antar-skenario internal
    """
    return {
        "MLI": float(np.mean(img_enhanced)),
        "CI": float(np.std(img_enhanced)),
        "Entropy": compute_entropy(img_enhanced),
        "AG": compute_average_gradient(img_enhanced),
        "SSIM": float(compute_ssim(img_ref, img_enhanced, data_range=1.0)),
        "PSNR": compute_psnr(img_enhanced, img_ref, max_val=pixel_max(img_ref)),
        "Noise_Est": estimate_noise_sigma(img_enhanced)
    }


def objective_function(img_temp: np.ndarray, img_in: np.ndarray,
                       e1: float, g1: float, l_max: float,
                       modified_sign: bool = False) -> float:
    """
    O_F = (V_T / mu_T) * ((E1 - E2) + (G1 - G2) / PSNR(I_T, I_in))   [Pers. 11, Dinh 2023]
    l_max = nilai maksimum piksel citra masukan (dihitung sekali di luar loop).
    """
    v_t = float(np.var(img_temp))
    mu_t = float(np.mean(img_temp))
    if mu_t <= 1e-8:
        return -1e9

    e2 = compute_entropy(img_temp)
    g2 = compute_average_gradient(img_temp)
    psnr_val = compute_psnr(img_temp, img_in, max_val=l_max)
    if psnr_val <= 1e-8:
        return -1e9

    if not modified_sign:
        term_e = (e1 - e2)
        term_g = (g1 - g2) / psnr_val
    else:
        term_e = (e2 - e1)
        term_g = (g2 - g1) / psnr_val

    return float((v_t / mu_t) * (term_e + term_g))


# ==============================================================================
# 3. DEKOMPOSISI GAUSSIAN & PEMROSESAN SUB-BAND
# ==============================================================================
def gaussian_decomposition(img_in: np.ndarray, kernel: np.ndarray) -> tuple:
    """GBF-TSID (Kieu et al., 2024): I_base = I_in * G ; I_detail = I_in - I_base."""
    i_base = convolve2d(img_in, kernel, mode='same', boundary='symm')
    i_detail = img_in - i_base
    return i_base, i_detail


def apply_clahe(img_01: np.ndarray, clip_limit: float, tile_grid_size: tuple) -> np.ndarray:
    """CLAHE pada rentang [0, 1]."""
    img_uint8 = np.round(np.clip(img_01, 0.0, 1.0) * 255.0).astype(np.uint8)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    return clahe.apply(img_uint8).astype(np.float64) / 255.0


def apply_led(img_01: np.ndarray, kernel: np.ndarray) -> np.ndarray:
    """
    Laplacian Edge Detection. |respons| SELALU dinormalisasi dengan nilai maksimumnya
    (rentang [0, 1]) di semua skenario, agar skala I_2 konsisten antara B dan C.
    """
    edge_mag = np.abs(convolve2d(img_01, kernel, mode='same', boundary='symm'))
    max_e = float(np.max(edge_mag))
    return edge_mag / max_e if max_e > 1e-12 else edge_mag


# ==============================================================================
# 4. MARINE PREDATORS ALGORITHM (MPA)
# ==============================================================================
def levy_flight(n: int, dim: int, beta: float = 1.5) -> np.ndarray:
    """Langkah acak berdistribusi Levy (R_L)."""
    num = math.gamma(1 + beta) * math.sin(math.pi * beta / 2)
    den = math.gamma((1 + beta) / 2) * beta * (2 ** ((beta - 1) / 2))
    sigma_u = (num / den) ** (1 / beta)
    u = np.random.normal(0, sigma_u, size=(n, dim))
    v = np.random.normal(0, 1, size=(n, dim))
    return 0.05 * u / (np.abs(v) ** (1 / beta))


def run_mpa_optimization(i_1: np.ndarray, i_2: np.ndarray, img_in: np.ndarray,
                         lb: np.ndarray, ub: np.ndarray, cfg: dict) -> tuple:
    """Mencari (w1*, w2*) untuk I_T = w1 * I_1 + w2 * I_2 dengan MPA (memaksimalkan O_F)."""
    n_prey = cfg["mpa_n_prey"]
    max_iter = cfg["mpa_max_iter"]
    K = cfg["mpa_K"]
    FADs = cfg["mpa_FADs"]
    dim = 2

    e1 = compute_entropy(img_in)
    g1 = compute_average_gradient(img_in)
    l_max = pixel_max(img_in)

    def evaluate_candidate(weights: np.ndarray) -> float:
        img_temp = np.clip(weights[0] * i_1 + weights[1] * i_2, 0.0, 1.0)
        return objective_function(img_temp, img_in, e1, g1, l_max, cfg["use_modified_fitness"])

    prey = lb + np.random.rand(n_prey, dim) * (ub - lb)
    fitness = np.array([evaluate_candidate(prey[i]) for i in range(n_prey)])

    best_idx = np.argmax(fitness)
    top_predator_pos = prey[best_idx].copy()
    top_predator_fit = fitness[best_idx]

    mem_prey = prey.copy()
    mem_fit = fitness.copy()

    for it in range(max_iter):
        for i in range(n_prey):
            prey[i] = np.clip(prey[i], lb, ub)
            fit_i = evaluate_candidate(prey[i])
            fitness[i] = fit_i
            if fit_i > top_predator_fit:
                top_predator_fit = fit_i
                top_predator_pos = prey[i].copy()

        improved = fitness > mem_fit
        mem_prey[improved] = prey[improved].copy()
        mem_fit[improved] = fitness[improved].copy()
        prey = mem_prey.copy()
        fitness = mem_fit.copy()

        elite = np.tile(top_predator_pos, (n_prey, 1))
        CF = (1.0 - (it / max_iter)) ** (2.0 * it / max_iter)

        R_B = np.random.randn(n_prey, dim)
        R_L = levy_flight(n_prey, dim, beta=1.5)
        R = np.random.rand(n_prey, dim)

        if it < max_iter / 3.0:                       # Tahap 1
            step_size = R_B * (elite - R_B * prey)
            prey = prey + K * R * step_size
        elif it < 2.0 * max_iter / 3.0:               # Tahap 2
            half = n_prey // 2
            step_1 = R_L[:half] * (elite[:half] - R_L[:half] * prey[:half])
            prey[:half] = prey[:half] + K * R[:half] * step_1
            step_2 = R_B[half:] * (R_B[half:] * elite[half:] - prey[half:])
            prey[half:] = elite[half:] + K * CF * step_2
        else:                                         # Tahap 3
            step_size = R_L * (R_L * elite - prey)
            prey = elite + K * CF * step_size

        r_fad = np.random.rand()                      # Eddy formation & FADs
        if r_fad < FADs:
            U = (np.random.rand(n_prey, dim) < FADs).astype(np.float64)
            prey = prey + CF * ((lb + np.random.rand(n_prey, dim) * (ub - lb)) * U)
        else:
            perm1 = np.random.permutation(n_prey)
            perm2 = np.random.permutation(n_prey)
            prey = prey + (FADs * (1.0 - r_fad) + r_fad) * (prey[perm1] - prey[perm2])

    for i in range(n_prey):
        prey[i] = np.clip(prey[i], lb, ub)
        fit_i = evaluate_candidate(prey[i])
        if fit_i > top_predator_fit:
            top_predator_fit = fit_i
            top_predator_pos = prey[i].copy()

    img_final = np.clip(top_predator_pos[0] * i_1 + top_predator_pos[1] * i_2, 0.0, 1.0)
    return img_final, top_predator_pos, top_predator_fit


# ==============================================================================
# 5. SKENARIO ABLATION INTERNAL (B, C, D)
# ==============================================================================
def run_ablation_scenario(img_in: np.ndarray, scenario: str, cfg: dict) -> dict:
    """
    'B': tanpa dekomposisi      -> I_1 = CLAHE(I_in),   I_2 = LED(I_in)
    'C': metode usulan          -> I_1 = CLAHE(I_base), I_2 = LED(I_detail)
    'D': tanpa LED pada detail  -> I_1 = CLAHE(I_base), I_2 = I_detail (mentah, bertanda +/-)
         Interpretasi D: detail layer TIDAK diproses LED, tetapi tetap dikalikan w2 dan
         dijumlahkan; hasil akhir di-clip ke [0, 1]. Tuliskan interpretasi ini di Bab III.
    """
    lb = cfg["bounds_weights"]["lb"]
    ub = cfg["bounds_weights"]["ub"]

    t0 = time.perf_counter()
    if scenario == "B":
        i_1 = apply_clahe(img_in, cfg["clahe_clip_limit"], cfg["clahe_tile_grid_size"])
        i_2 = apply_led(img_in, cfg["laplacian_kernel"])
    elif scenario == "C":
        i_base, i_detail = gaussian_decomposition(img_in, cfg["gaussian_kernel"])
        i_1 = apply_clahe(i_base, cfg["clahe_clip_limit"], cfg["clahe_tile_grid_size"])
        i_2 = apply_led(i_detail, cfg["laplacian_kernel"])
    elif scenario == "D":
        i_base, i_detail = gaussian_decomposition(img_in, cfg["gaussian_kernel"])
        i_1 = apply_clahe(i_base, cfg["clahe_clip_limit"], cfg["clahe_tile_grid_size"])
        i_2 = i_detail
    else:
        raise ValueError(f"Skenario '{scenario}' tidak valid. Gunakan 'B', 'C', atau 'D'.")
    t_pre = time.perf_counter() - t0

    t1 = time.perf_counter()
    img_enh, w_best, f_best = run_mpa_optimization(i_1, i_2, img_in, lb, ub, cfg)
    t_opt = time.perf_counter() - t1

    m = evaluate_all_metrics(img_enh, img_in)
    m.update({
        "Fitness": f_best,
        "w1_opt": float(w_best[0]),
        "w2_opt": float(w_best[1]),
        "T_preprocessing_sec": t_pre,
        "T_optimization_sec": t_opt,
        "T_total_sec": t_pre + t_opt,
        "Enhanced_Image": img_enh
    })
    return m


# ==============================================================================
# 6. EKSEKUSI END-TO-END
# ==============================================================================
def load_mri_image(filepath: str, target_size: tuple) -> np.ndarray:
    img = cv2.imread(filepath, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(f"Gagal membaca file citra: {filepath}")
    if img.shape != target_size:
        img = cv2.resize(img, target_size, interpolation=cv2.INTER_CUBIC)
    return img.astype(np.float64) / 255.0


def main():
    os.makedirs(CONFIG["output_dir"], exist_ok=True)

    exts = ("*.png", "*.jpg", "*.jpeg", "*.tif", "*.tiff", "*.bmp")
    image_paths = []
    for ext in exts:
        image_paths.extend(glob.glob(os.path.join(CONFIG["dataset_dir"], "**", ext), recursive=True))
    image_paths = sorted(image_paths)

    if len(image_paths) == 0:
        print(f"[INFO] Folder '{CONFIG['dataset_dir']}' kosong. Membuat citra uji sintetis 256x256...")
        os.makedirs(CONFIG["dataset_dir"], exist_ok=True)
        dummy_path = os.path.join(CONFIG["dataset_dir"], "sample_MRI_070T.png")
        np.random.seed(CONFIG["random_seed"])
        y, x = np.ogrid[:256, :256]
        phantom = np.exp(-((x - 128) ** 2 / (2 * 60 ** 2) + (y - 128) ** 2 / (2 * 80 ** 2)))
        phantom += 0.03 * np.random.randn(256, 256)
        cv2.imwrite(dummy_path, np.clip(phantom * 200, 0, 255).astype(np.uint8))
        image_paths = [dummy_path]

    scenarios = ["B", "C", "D"]
    records = []
    print(f"Memulai eksperimen pada {len(image_paths)} citra untuk Skenario {scenarios}...\n")

    for idx, img_path in enumerate(image_paths, start=1):
        img_name = os.path.basename(img_path)
        img_in = load_mri_image(img_path, CONFIG["image_size"])
        print(f"[{idx}/{len(image_paths)}] Memproses: {img_name}")

        # Baris NE: hanya MLI, CI, Entropy, AG (+ noise) yang bermakna
        ne = evaluate_all_metrics(img_in, img_in)
        ne.update({
            "Image": img_name, "Scenario": "Original (NE)",
            "SSIM": np.nan, "PSNR": np.nan,
            "Fitness": np.nan, "w1_opt": np.nan, "w2_opt": np.nan,
            "T_preprocessing_sec": np.nan, "T_optimization_sec": np.nan, "T_total_sec": np.nan
        })
        records.append(ne)

        for sc in scenarios:
            np.random.seed(CONFIG["random_seed"] + idx)   # aliran acak sama untuk B, C, D
            res = run_ablation_scenario(img_in, sc, CONFIG)
            enhanced = res.pop("Enhanced_Image")

            folder = os.path.join(CONFIG["output_dir"], f"Skenario_{sc}")
            os.makedirs(folder, exist_ok=True)
            out_path = os.path.join(folder, f"{os.path.splitext(img_name)[0]}_Skenario_{sc}.png")
            cv2.imwrite(out_path, np.clip(np.round(enhanced * 255.0), 0, 255).astype(np.uint8))

            res["Image"] = img_name
            res["Scenario"] = f"Skenario {sc}"
            records.append(res)

            print(
                f"   -> Skenario {sc} | MLI: {res['MLI']:.4f} | CI: {res['CI']:.4f} | "
                f"E: {res['Entropy']:.4f} | AG: {res['AG']:.4f} | SSIM: {res['SSIM']:.4f} | "
                f"PSNR: {res['PSNR']:.2f} dB | Noise: {res['Noise_Est']:.4f} | "
                f"w1={res['w1_opt']:.3f}, w2={res['w2_opt']:.3f} | Waktu: {res['T_total_sec']:.3f}s"
            )

    df = pd.DataFrame(records)
    cols = ["Image", "Scenario", "MLI", "CI", "Entropy", "AG", "SSIM", "PSNR", "Noise_Est",
            "Fitness", "w1_opt", "w2_opt", "T_preprocessing_sec", "T_optimization_sec", "T_total_sec"]
    df = df[cols]
    detail_path = os.path.join(CONFIG["output_dir"], "Hasil_Detail_Per_Citra.csv")
    df.to_csv(detail_path, index=False)

    num_cols = cols[2:]
    summary = df.groupby("Scenario", sort=False)[num_cols].mean().reset_index()

    # Referensi Tabel 9 Dinh (2023), dataset M1 (hanya 4 metrik; tidak ada SSIM/PSNR/waktu)
    dinh = pd.DataFrame([
        {"Scenario": "Dinh (2023) - NE [Paper]",       "MLI": 0.3045, "CI": 0.3203, "Entropy": 4.6393, "AG": 0.0803},
        {"Scenario": "Dinh (2023) - CLAHE [Paper]",    "MLI": 0.3358, "CI": 0.3345, "Entropy": 5.6665, "AG": 0.0910},
        {"Scenario": "Dinh (2023) - EFF [Paper]",      "MLI": 0.3604, "CI": 0.3619, "Entropy": 5.5792, "AG": 0.0927},
        {"Scenario": "Dinh (2023) - EGIF [Paper]",     "MLI": 0.3147, "CI": 0.3675, "Entropy": 4.9362, "AG": 0.1181},
        {"Scenario": "Dinh (2023) - FFM [Paper]",      "MLI": 0.3447, "CI": 0.3391, "Entropy": 5.9453, "AG": 0.0884},
        {"Scenario": "Dinh (2023) - Baseline [Paper]", "MLI": 0.3601, "CI": 0.3921, "Entropy": 6.0349, "AG": 0.1459},
    ])
    combined = pd.concat([summary, dinh], ignore_index=True)
    summary_path = os.path.join(CONFIG["output_dir"], "Ringkasan_Perbandingan_Bab4.csv")
    combined.to_csv(summary_path, index=False)

    print("\n" + "=" * 95)
    print("RINGKASAN HASIL & PERBANDINGAN DENGAN TABEL 9 DINH (2023):")
    print("=" * 95)
    print(combined.to_string(index=False))
    print(f"\nDetail per citra : {detail_path}")
    print(f"Ringkasan Bab IV : {summary_path}")


if __name__ == "__main__":
    main()

Memulai eksperimen pada 89 citra untuk Skenario ['B', 'C', 'D']...

[1/89] Memproses: 25055.png
   -> Skenario B | MLI: 0.4709 | CI: 0.4357 | E: 4.4669 | AG: 0.0961 | SSIM: 0.5446 | PSNR: 13.62 dB | Noise: 0.0239 | w1=1.500, w2=0.500 | Waktu: 2.093s
   -> Skenario C | MLI: 0.4683 | CI: 0.4239 | E: 4.9881 | AG: 0.0653 | SSIM: 0.4898 | PSNR: 13.27 dB | Noise: 0.0140 | w1=1.479, w2=0.499 | Waktu: 1.522s
   -> Skenario D | MLI: 0.4544 | CI: 0.4210 | E: 4.9690 | AG: 0.0658 | SSIM: 0.5239 | PSNR: 14.06 dB | Noise: 0.0075 | w1=1.500, w2=0.103 | Waktu: 2.104s
[2/89] Memproses: 25056.png
   -> Skenario B | MLI: 0.4678 | CI: 0.4353 | E: 4.4925 | AG: 0.0969 | SSIM: 0.5478 | PSNR: 13.73 dB | Noise: 0.0244 | w1=1.500, w2=0.499 | Waktu: 1.922s
   -> Skenario C | MLI: 0.4690 | CI: 0.4249 | E: 4.9743 | AG: 0.0649 | SSIM: 0.4897 | PSNR: 13.22 dB | Noise: 0.0139 | w1=1.476, w2=0.500 | Waktu: 1.702s
   -> Skenario D | MLI: 0.4529 | CI: 0.4239 | E: 4.9365 | AG: 0.0747 | SSIM: 0.5449 | PSNR: 14.24 dB | Noi

In [3]:
import pandas as pd
import os

# 1. Memuat file hasil detail per citra
output_dir = "./Hasil_Eksperimen"  # Sesuaikan jika path folder output-mu berbeda
detail_path = os.path.join(output_dir, "Hasil_Detail_Per_Citra.csv")

if os.path.exists(detail_path):
    df_detail = pd.read_csv(detail_path)
    
    # 2. Membuat Tabel Rata-rata Skenario Internal (B, C, D) & Original
    cols_internal = ["Scenario", "MLI", "CI", "Entropy", "AG", "SSIM", "PSNR", "Noise_Est", "T_total_sec"]
    summary_internal = df_detail.groupby("Scenario", sort=False)[cols_internal[1:]].mean().reset_index()
    
    print("=" * 110)
    print(" TABEL 1: PERBANDINGAN KINERJA INTERNAL (ABLASI STUDY)")
    print("=" * 110)
    display(summary_internal.round(4))
    
    # Keterangan panduan arah metrik di bawah tabel 1
    print("\n📌 PANDUAN ARAH METRIK (TABEL 1):")
    print(" • MLI (Mean Level Intensity)   : Semakin tinggi menunjukkan citra lebih terang.")
    print(" • CI (Contrast Index)          : Semakin tinggi semakin bagus (kontras & detail jelas).")
    print(" • Entropy                      : Semakin tinggi semakin bagus (informasi/tekstur makin kaya).")
    print(" • AG (Average Gradient)        : Semakin tinggi semakin bagus (ketajaman tepi meningkat).")
    print(" • SSIM                         : Semakin tinggi semakin bagus (kemiripan struktur dengan asli).")
    print(" • PSNR                         : Semakin tinggi semakin bagus (kualitas sinyal/minim distorsi).")
    print(" • Noise_Est                    : SEMAKIN RENDAH SEMAKIN BAGUS (makin bersih dari noise).")
    print(" • T_total_sec                  : Semakin rendah semakin efisien (waktu komputasi cepat).")
    
    # 3. Penjelasan Skenario B, C, dan D untuk naskah/catatan
    print("\n" + "=" * 110)
    print(" KETERANGAN SKENARIO ABLASI:")
    print("=" * 110)
    print(" • Skenario B : Tanpa Dekomposisi Gaussian (CLAHE + LED langsung ke seluruh citra utuh).")
    print(" • Skenario C : Metode Usulan Utama (Dekomposisi Gaussian + CLAHE di Base Layer + LED di Detail Layer + MPA).")
    print(" • Skenario D : Skenario Ablasi Tanpa LED (Dekomposisi Gaussian + CLAHE di Base Layer + Detail Mentah tanpa LED + MPA).")

    # 4. Membuat Tabel Perbandingan Eksternal dengan Baseline Dinh (2023)
    cols_external = ["Scenario", "MLI", "CI", "Entropy", "AG"]
    summary_external = df_detail.groupby("Scenario", sort=False)[cols_external[1:]].mean().reset_index()
    
    dinh_paper = pd.DataFrame([
        {"Scenario": "Dinh (2023) - NE [Paper]",       "MLI": 0.3045, "CI": 0.3203, "Entropy": 4.6393, "AG": 0.0803},
        {"Scenario": "Dinh (2023) - CLAHE [Paper]",    "MLI": 0.3358, "CI": 0.3345, "Entropy": 5.6665, "AG": 0.0910},
        {"Scenario": "Dinh (2023) - EFF [Paper]",      "MLI": 0.3604, "CI": 0.3619, "Entropy": 5.5792, "AG": 0.0927},
        {"Scenario": "Dinh (2023) - EGIF [Paper]",     "MLI": 0.3147, "CI": 0.3675, "Entropy": 4.9362, "AG": 0.1181},
        {"Scenario": "Dinh (2023) - FFM [Paper]",      "MLI": 0.3447, "CI": 0.3391, "Entropy": 5.9453, "AG": 0.0884},
        {"Scenario": "Dinh (2023) - Baseline [Paper]", "MLI": 0.3601, "CI": 0.3921, "Entropy": 6.0349, "AG": 0.1459},
    ])
    
    combined_table = pd.concat([summary_external, dinh_paper], ignore_index=True)
    
    print("\n" + "=" * 110)
    print(" TABEL 2: PERBANDINGAN EKSTERNAL DENGAN METODE LAIN (TABEL 9 DINH, 2023)")
    print("=" * 110)
    display(combined_table.round(4))
    
    # Simpan tabel ringkasan gabungan
    combined_table.to_csv(os.path.join(output_dir, "Tabel_Perbandingan_Bab4.csv"), index=False)
    print(f"\nTabel ringkasan berhasil disimpan ke: {os.path.join(output_dir, 'Tabel_Perbandingan_Bab4.csv')}")
else:
    print(f"[ERROR] File '{detail_path}' tidak ditemukan. Pastikan cell eksekusi utama sudah dijalankan sampai selesai!")

 TABEL 1: PERBANDINGAN KINERJA INTERNAL (ABLASI STUDY)


,Scenario,MLI,CI,Entropy,AG,SSIM,PSNR,Noise_Est,T_total_sec
0,Original (NE),0.3041,0.3204,4.6346,0.0809,NaN,NaN,0.0146,NaN
1,Skenario B,0.3980,0.3687,4.8332,0.0796,0.5932,13.8166,0.0164,1.4759
2,Skenario C,0.2711,0.2451,5.4125,0.0419,0.6182,13.3978,0.0061,1.5354
3,Skenario D,0.2934,0.2733,5.3368,0.0481,0.6269,13.8189,0.0048,1.4787



📌 PANDUAN ARAH METRIK (TABEL 1):
 • MLI (Mean Level Intensity)   : Semakin tinggi menunjukkan citra lebih terang.
 • CI (Contrast Index)          : Semakin tinggi semakin bagus (kontras & detail jelas).
 • Entropy                      : Semakin tinggi semakin bagus (informasi/tekstur makin kaya).
 • AG (Average Gradient)        : Semakin tinggi semakin bagus (ketajaman tepi meningkat).
 • SSIM                         : Semakin tinggi semakin bagus (kemiripan struktur dengan asli).
 • PSNR                         : Semakin tinggi semakin bagus (kualitas sinyal/minim distorsi).
 • Noise_Est                    : SEMAKIN RENDAH SEMAKIN BAGUS (makin bersih dari noise).
 • T_total_sec                  : Semakin rendah semakin efisien (waktu komputasi cepat).

 KETERANGAN SKENARIO ABLASI:
 • Skenario B : Tanpa Dekomposisi Gaussian (CLAHE + LED langsung ke seluruh citra utuh).
 • Skenario C : Metode Usulan Utama (Dekomposisi Gaussian + CLAHE di Base Layer + LED di Detail Layer + MPA).
 • Sken

,Scenario,MLI,CI,Entropy,AG
0,Original (NE),0.3041,0.3204,4.6346,0.0809
1,Skenario B,0.3980,0.3687,4.8332,0.0796
2,Skenario C,0.2711,0.2451,5.4125,0.0419
3,Skenario D,0.2934,0.2733,5.3368,0.0481
4,Dinh (2023) - NE [Paper],0.3045,0.3203,4.6393,0.0803
5,Dinh (2023) - CLAHE [Paper],0.3358,0.3345,5.6665,0.0910
6,Dinh (2023) - EFF [Paper],0.3604,0.3619,5.5792,0.0927
7,Dinh (2023) - EGIF [Paper],0.3147,0.3675,4.9362,0.1181
8,Dinh (2023) - FFM [Paper],0.3447,0.3391,5.9453,0.0884
9,Dinh (2023) - Baseline [Paper],0.3601,0.3921,6.0349,0.1459



Tabel ringkasan berhasil disimpan ke: ./Hasil_Eksperimen/Tabel_Perbandingan_Bab4.csv
